<a href="https://colab.research.google.com/github/dave-heslop74/EMSC2010-W8-L1/blob/main/EMSC2010_W8_L1_NB1_uXXXXXXX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010-W8-L1
In this notebook we'll use the river grain size data to investigate how to estimate the sample correlation coefficient $r$. We'll then use Bayes theorem and ```PyMC``` to sample the posterior probability distribution of the population correlation coefficient $\rho$.

First load the packages we'll need.

In [ ]:
import numpy as np #for working with numerical arrays
import matplotlib.pyplot as plt #for plotting
import pymc as pm #for automated Bayesian inference
import arviz as az #for analysis of Bayesian models

We can now input the data

In [ ]:
dist = np.array([1, 1.25, 1.75, 2.25, 2.5, 3,  3.25, 3.75, 4.25, 4.5, 4.75, 5, 5.4, 6, 6.35, 7.1, 7.5, 7.9, 8.5, 8.7, 9.1, 9.6, 10.2]) #distance downstream in km
gs = np.array([-10.5411, -10.8376, -10.388, -10.3219, -10.4818, -10.388, -10.1421, -10.0084, -10.0084, -9.9366, -9.5699, -9.7977, -9.6257, -9.388, -9.2527, -9.388, -9.1033, -9.1799, -9.0224, -8.8455, -9.1033, -8.9366, -9.1799]) #grainsize in phi

Plot the data to provide an initial visualization.

In [ ]:
plt.plot(dist,gs,'ok')
plt.xlabel('Distance (km)')
plt.ylabel('Grain size (phi)')
plt.minorticks_on()

As we discussed on the whiteboard, to find the correlation coefficient, *r*, we first calculate the deviations for the *x* and *y* variables.

$x = X-\bar{X}$,

$y = Y-\bar{Y}$,

We'll also plot the deviations to demonstrate how the calculation of $r$ works.

In [ ]:
x = dist - np.mean(dist) #deviation in the x variable
y = gs - np.mean(gs) #deviation in the y variable
plt.plot(x,y,'ok') #plot the deviations
plt.gca().axvline(x=0, color = 'k', linestyle = '--') #add a vertical line at x=0
plt.gca().axhline(y=0, color = 'k', linestyle = '--') #add a horizontal line at y=0
plt.xlabel('Distance deviation (km)') #label the x-axis
plt.ylabel('Grain size deviation (phi)') #label the y-axis
plt.minorticks_on() #include minor ticks

Remember that $r$ is given by:

## $\frac{\Sigma x y}{\sqrt{\Sigma x^2} \sqrt{\Sigma y^2}}$

where $\Sigma$ indicates that we should add up all the values.

In [ ]:
num = np.sum(x*y) #find the numerator
den = np.sqrt(np.sum(x**2)) * np.sqrt(np.sum(y**2)) #find the denominator
r = num/den #obtain r
print('Sample correlation r = {:.2f}'.format(r)) #print result to 2 d.p.

In the equation to find $r$, you can see that the numerator involves multiplying the $x$ and $y$ deviations. This multiplication provides a measure of how closely the two variables move together. When the deviations of an observation share the same sign (indicating the first or third quadrant), the product $xy$ is positive. Conversely, if the deviations have opposite signs (indicating the second or fourth quadrant), the product $xy$ is negative. In cases where $X$ and $Y$ show a positive relationship (i.e., $Y$ increases as $X$ increases) most deviations define points in the first or third quadrants, resulting in a positive $\Sigma xy$. Conversely, in negative relationships, where one variable decreases as the other increases, most deviations define points in the second or fourth quadrants, yielding a negative $\Sigma xy$. When no relationship exists between $X$ and $Y$, deviations are spread across all four quadrants, causing the signs of $xy$ to cancel out, resulting in a $\Sigma xy$ close to zero.

The sign of $\Sigma xy$ indicates whether the correlation is positive or negative, but there's an issue with its magnitude. Each $xy$ value depends on the units of the data. For instance, in our river gravel example, if downstream distance were measured in millimetres instead of kilometres, each $xy$ value would be 10$^6$ times larger because 10$^6$ millimetres equal 1 kilometre. To address this, we consider the magnitudes of $x$ and $y$ in the denominator of the equation, making the $r$ "scale invariant."

## Applying Bayes' theorem to infer $\rho$

Ultimately, we are interested in inferring the population correlation coefficient $\rho$. We can do this using Bayes' theorem to estimate the bivariate normal distribution that corresponds to the relationship between distance and grain size. As we discussed on the whiteboard, a bivariate normal distribution is defined by the mean:

\begin{bmatrix}
\mu_x \\
\mu_y
\end{bmatrix}

and covariance matrix:
\begin{bmatrix}
\sigma_x^2 & \rho \sigma_x \sigma_y\\
\rho \sigma_x \sigma_y & \sigma_y^2
\end{bmatrix}

We need to estimate all of these terms and therefore we need to assign them all priors in ```PyMC```.

In [ ]:
data = np.column_stack([dist, gs]) #form a variable with dist in the first column and gs in the second column (this is needed by PyMC)

with pm.Model() as model: #define the PyMC model

    # Priors for means of the bivariate normal distribution
    mu = pm.Normal("mu", mu=0, sigma=5, shape=2)

    # Prior for correlation coefficient — uniform between -1 and 1
    rho = pm.Uniform("rho", lower=-1, upper=1)

    # Prior for standard deviations of the bivariate normal distribution
    sigma = pm.HalfNormal("sigma", sigma=2, shape=2)

    # Build the covariance matrix from rho and sigmas
    cov = pm.math.stack([
        [sigma[0]**2, rho * sigma[0] * sigma[1]],
        [rho * sigma[0] * sigma[1], sigma[1]**2             ]
    ])

    # Likelihood is a "multivariate" (in this case bivariate) normal distribution
    obs = pm.MvNormal("obs", mu=mu, cov=cov, observed=data)

    # Obtain 5000 samples from the posterior distributions (mu, rho, & sigma)
    trace = pm.sample(sample=5000)

The ```Arviz``` package can also provide us with a summary of the posterior, such as the 95% credible intervals:

In [ ]:
az.plot_posterior(trace, var_names=["rho"])
az.summary(trace, var_names=["rho"], hdi_prob=0.95)

If the 95% credible interval does not include $\rho = 0$ we can consider the population to have a significant correlation. In other words, there is a linear relationship between $x$ and $y$.

However, the analysis **does not** tell us **why** there is a relationship (think back to our chocolate and intelligence hypothesis).